# E. coli fed-batch: native forward simulation
Direct configured kinetics; no fitting or reconciliation. See README for basis and limitations.


In [1]:
# STEP 1: Select runs and folders
# WHAT TO DO: Use Run All or Shift+Enter in order. Copy the input folders to configure
# another process; change INPUT_DIRS below. No measurements are needed to simulate.
from pathlib import Path
import json
import csv
import numpy as np
import matplotlib.pyplot as plt
import sys
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'PharmaPy').is_dir())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from PharmaPy.Bioreactors import build_bioreactor, BioreactorUnitConverter as Units

EXAMPLE = ROOT / 'examples/bioreactors/ecoli_ye_fed_batch'
INPUT_DIRS = [EXAMPLE / 'inputs', EXAMPLE / 'inputs/F11']
THERMO = EXAMPLE / 'inputs/thermo.json'
EXPORT_DIR = EXAMPLE / 'outputs'
WRITE_ARTIFACTS = True


In [2]:
# STEP 2: Load process configurations
# WHAT TO DO: Configure equations and initial amounts in mechanism.json; case.json selects
# runtime and numerics. Quantities declare value/unit; legacy recipe suffixes remain supported.
# See README for concentration and rate declarations. Each feed appears once.
configurations = [(json.loads((p / 'case.json').read_text()),
                   json.loads((p / 'mechanism.json').read_text())) for p in INPUT_DIRS]
thermo = json.loads(THERMO.read_text())


In [3]:
# STEP 3: Build native reactors from the configured inputs
# WHAT TO DO: No process-specific Python is needed. Configure kinetics and recipes in JSON.
# build_bioreactor validates inputs and attaches the shared native mechanisms and streams.
assemblies = {case['recipe']['name']: build_bioreactor(case, config, str(THERMO))
              for case, config in configurations}


In [4]:
# STEP 4: Forward simulate through native solve_unit
# WHAT TO DO: Run each assembly. The shared workflow applies flow changes, additions and
# well-mixed samples at scheduled boundaries. Rebuild in Step 3 before rerunning a simulation.
solutions = {}
for run, assembly in assemblies.items():
    histories = assembly.solve(verbose=False)
    solutions[run] = (assembly.phase, histories)
    print(run, type(assembly.unit).__name__, len(histories), 'segments completed')


F10 SemiBatchReactor 97 segments completed
F11 SemiBatchReactor 92 segments completed


In [5]:
# STEP 5: Tabulate and plot trajectories
# WHAT TO DO: All concentrations come from native inventories divided by native working
# volume. Duplicate boundary times preserve pre/post event states. Edit plotting labels here.
tables, figures = {}, {}
for run, (phase, histories) in solutions.items():
    rows = []
    for result in histories:
        for t, mass, vol in zip(result.time, result.mass_j_liquid0, np.asarray(result.vessel_vol).reshape(-1)):
            row = {'time_h': Units.convert(t, 's', 'h'), 'volume_l': Units.convert(vol, 'm3', 'L')}
            row.update({n+'_g_l': Units.convert(m/vol, 'kg/m3', 'g/L') for n, m in zip(phase.name_species, mass)})
            row.update({n+'_g': Units.convert(m, 'kg', 'g') for n, m in zip(phase.name_species, mass)})
            rows.append(row)
    tables[run] = rows
    quantities = ['volume_l'] + [n+'_g_l' for n in phase.name_species if n != assemblies[run].carrier_species]
    fig, axes = plt.subplots(2, int(np.ceil(len(quantities)/2)), figsize=(13,7), constrained_layout=True)
    for ax, name in zip(axes.flat, quantities):
        ax.plot([r['time_h'] for r in rows], [r[name] for r in rows])
        ax.set(xlabel='Time (h)', ylabel=name); ax.grid(alpha=.25)
    fig.suptitle(run+' — native PharmaPy, single-counted feed')
    figures[run] = fig
    print(run, 'final volume (L):', rows[-1]['volume_l'])
plt.show()


F10 final volume (L): 1.1060999999999996
F11 final volume (L): 1.100799999999996


In [6]:
# STEP 6: Export forward trajectories and figures
# WHAT TO DO: Run to write CSV spreadsheets and PNG/SVG figures. Set WRITE_ARTIFACTS=False
# in Step 1 for a calculation without replacing files.
if WRITE_ARTIFACTS:
    for run, rows in tables.items():
        folder = EXPORT_DIR / run
        folder.mkdir(parents=True, exist_ok=True)
        with (folder/'trajectories.csv').open('w') as stream:
            writer = csv.DictWriter(stream, fieldnames=list(rows[0]), lineterminator='\n')
            writer.writeheader(); writer.writerows(rows)
        figures[run].savefig(folder/'trajectories.png', dpi=160)
        figures[run].savefig(folder/'trajectories.svg')
